# 2. Compare fixed and retrained models

Would refitting a return model every quarter improve the stocks it selects? We'll choose Ridge's penalty using development data, then compare a model fitted once with one updated quarterly. Both use the same portfolio rules, so differences in their results come from the forecasts.

The examples use the **30-stock class extract** and evaluate **2015**. HW4 extends this work to the full stock universe and the April 2016–September 2026 final period.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
import sys
%matplotlib inline
course_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
modeling_dir = course_root / 'Lectures/modeling'
if str(modeling_dir) not in sys.path:
    sys.path.insert(0, str(modeling_dir))
from course_data import (load_membership, apply_membership, load_data, prepare_panel, price_features, financial_events,
                        join_financials, make_labels, date_folds, make_pipeline,
                        prediction_metrics, QuantileClipper, PRICE_FEATURES,
                        FINANCIAL_FEATURES, NUMERIC_FEATURES, FEATURES)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

from backtest import run_backtest, performance

In [ ]:
from sklearn.base import clone
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
import sklearn
import duckdb
import json
prices, fundamentals, universe, benchmark = load_data()
panel = prepare_panel()
calendar = pd.DatetimeIndex(benchmark.date)
fit_date = pd.Timestamp('2014-12-31')
development = panel.loc[panel.eligible & panel.date.ge('2010-01-01')
                        & panel.date.lt(fit_date) & panel.label_end.lt(fit_date)].reset_index(drop=True)
folds = date_folds(development, validation_years=(2012, 2013, 2014))

## 1. Choose the rules before evaluating 2015

We want to compare two ways of using the same model: fit it once, or update it quarterly as more outcomes become known. To make that comparison interpretable, first fix the features, Ridge penalty, stock-selection rule, holding period, and costs. Otherwise an improvement could come from changing several decisions at once.

The configuration below records those choices. Ridge's `alpha` hyperparameter is selected by mean RMSE over the 2012, 2013, and 2014 validation folds. None of the 2015 outcomes participates in that search. Both portfolio versions then buy the three highest-ranked eligible stocks each month under the same cash and holding rules.

Save the database and package versions too. A random seed controls random draws, but another researcher using revised source data or a different software version may still get different results.

In [ ]:
experiment = {
    'name': 'class_ridge_2015', 'database': 'equities_class.duckdb',
    'features': FEATURES, 'training_signal_start': '2010-01-01',
    'initial_fit_signal': str(fit_date.date()), 'validation_years': [2012, 2013, 2014],
    'selection_metric': 'mean fold RMSE', 'alpha_grid': [1, 100, 1000],
    'test_start': '2015-01-01', 'test_end': '2015-12-31',
    'top_n': 3, 'cohort_fraction': 1/3, 'holding_calendar_days': 90,
    'fee_bps_per_side': 5, 'initial_cash': 1_000_000,
    'cash_yield': 0, 'risk_free_hurdle': 0.02,
    'universe_policy': 'quarterly membership for next-session entries; never filter held-price history',
    'rolling_fit_signals': ['2014-12-31', '2015-03-31', '2015-06-30', '2015-09-30'],
    'versions': {'pandas': pd.__version__, 'numpy': np.__version__,
                 'sklearn': sklearn.__version__, 'duckdb': duckdb.__version__},
}
with duckdb.connect(str(modeling_dir / 'data/equities_class.duckdb'), read_only=True) as con:
    experiment['dataset_metadata'] = dict(con.execute('SELECT key,value FROM dataset_metadata').fetchall())
display(pd.Series({k: v for k, v in experiment.items() if k not in ['features', 'dataset_metadata']}))
search = GridSearchCV(make_pipeline(Ridge()), {'model__alpha': experiment['alpha_grid']},
                      cv=folds, scoring='neg_root_mean_squared_error', n_jobs=1, error_score='raise')
search.fit(development[FEATURES], development.target)
experiment['chosen_alpha'] = search.best_params_['model__alpha']
print('Frozen alpha:', experiment['chosen_alpha'])

## 2. Fit once and produce the full set of predictions

The fixed model is fitted at the close of December 31, 2014 using only completed outcomes ending before that date. It retains the same fitted coefficients, scaling statistics, clipping bounds, and imputation values throughout 2015. This asks how well the relationships learned before 2015 continue to work.

Include December 31 when generating scores, because those forecasts determine the first purchases in the test period. Keep rows whose future targets are missing: at prediction time we know the current features, not whether an exit quote will still be available months later. The output table records the signal date, ticker, forecast, and model fit date.

`test_features[FEATURES]` uses the same explicit input list as training. Selecting every column except `target` would also pass columns such as `label_end`, which reveal something about the future and must stay out of the model.

In [ ]:
test_features = panel.loc[panel.eligible & panel.date.ge(fit_date)].copy()
fixed_model = search.best_estimator_
fixed_scores = test_features[['date', 'symbol']].copy()
fixed_scores['score'] = fixed_model.predict(test_features[FEATURES])
fixed_scores['fit_date'] = fit_date
assert development.label_end.max() < fixed_scores.date.min()
assert fixed_scores.fit_date.le(fixed_scores.date).all()
print('Scored rows:', len(fixed_scores), '; scored rows without observed targets:', test_features.target.isna().sum())
display(fixed_scores.head())

## 3. Refit quarterly as new outcomes become known

A fixed model can become stale if relationships change. A **walk-forward** model addresses that by fitting again on scheduled dates and using the updated fit only for subsequent forecasts. Here we refit before the January, April, July, and October cohorts.

At the June 30 fit, some early-2015 signals have completed their 90-day holding periods, so their realized returns can now join training. A signal from late June is still unresolved and cannot. The filter `label_end < asof` makes that distinction separately for every row. Filtering only the feature date would accidentally train on outcomes that lie in the future of that fit.

Our training window expands from 2010 as new completed observations accumulate. The code's variables call this `rolling`, but it does not discard older years; a fixed-length rolling window would be another possible development choice.

Using newly completed test-period outcomes in these scheduled fits is part of the rule chosen before testing. Choosing a new feature or penalty after seeing which one improves 2015 performance would instead use the test results to redesign the strategy. Here the feature definitions and selected penalty remain fixed, while coefficients and preprocessing are refitted.

Inspect `fit_audit`: for each fit, the last training label must end before its first scored signal. Each prediction also carries `fit_date`, so we can identify exactly which fitted model produced it.

In [ ]:
rolling_parts, fit_audit = [], []
fit_signals = pd.to_datetime(experiment['rolling_fit_signals'])
for i, asof in enumerate(fit_signals):
    next_asof = fit_signals[i+1] if i+1 < len(fit_signals) else pd.Timestamp('2016-01-01')
    available = panel.loc[panel.eligible & panel.date.ge('2010-01-01')
                          & panel.date.lt(asof) & panel.label_end.lt(asof)]
    to_score = panel.loc[panel.eligible & panel.date.ge(asof) & panel.date.lt(next_asof)]
    model = clone(fixed_model).fit(available[FEATURES], available.target)
    predicted = to_score[['date', 'symbol']].copy()
    predicted['score'] = model.predict(to_score[FEATURES])
    predicted['fit_date'] = asof
    rolling_parts.append(predicted)
    fit_audit.append({'fit_date': asof, 'rows': len(available),
                      'last_training_signal': available.date.max(),
                      'last_label_end': available.label_end.max(),
                      'first_scored_signal': to_score.date.min(), 'last_scored_signal': to_score.date.max()})
    assert available.label_end.max() < to_score.date.min()
rolling_scores = pd.concat(rolling_parts, ignore_index=True)
fit_audit = pd.DataFrame(fit_audit)
assert not rolling_scores.duplicated(['date', 'symbol']).any()
display(fit_audit)

## 4. Compare forecasts under the same trading rules

Feed the fixed model, the quarterly-refitted model, and a 63-session momentum ranking into the same backtester. They receive the same initial cash and follow the same entry dates, holding periods, fees, and eligibility rules. Differences between these runs therefore come from their scores and the trades those scores cause.

The S&P 500 total-return benchmark starts at the same first test close. It provides a familiar market comparison, but it owns a much broader portfolio than our three-stock cohorts. The momentum strategy is a closer comparison for the question of whether fitting a model improved our stock selection.

The engine receives only `date, symbol, score`. A model-fitting error should stop the experiment and be investigated. Catching that error and substituting cash would quietly make the portfolio follow a different rule on the affected dates.

In [ ]:
momentum_scores = test_features[['date', 'symbol', 'momentum_63']].rename(columns={'momentum_63': 'score'})
score_sets = {'fixed Ridge': fixed_scores, 'rolling Ridge': rolling_scores, '63-session momentum': momentum_scores}
runs = {}
for name, scores in score_sets.items():
    runs[name] = run_backtest(prices, benchmark, scores[['date', 'symbol', 'score']],
                             start='2015-01-01', end='2015-12-31', fee_bps=5)
comparison = pd.DataFrame({name: performance(run['history'].nav) for name, run in runs.items()})
comparison['S&P 500 total return'] = performance(runs['fixed Ridge']['history'].benchmark_nav)
display(comparison)
fig, ax = plt.subplots(figsize=(10, 5))
for name, run in runs.items():
    run['history'].nav.div(1_000_000).sub(1).plot(ax=ax, label=name)
runs['fixed Ridge']['history'].benchmark_nav.div(1_000_000).sub(1).plot(ax=ax, label='S&P 500 total return', color='black', ls='--')
ax.set(ylabel='Cumulative net return', title='2015 classroom evaluation: identical accounting, different scores')
ax.legend()
plt.tight_layout()

Start by comparing the score rankings and the fills, not just the two NAV curves. Refitting can change predicted return levels without changing the top three stocks. Even a changed ranking may produce no purchase when cash is tied up in older lots. These cases explain how different models can lead to the same portfolio path.

### Check prediction quality as a separate result

Prediction metrics ask how close forecasts were to observed returns. Portfolio metrics ask what happened to the money under our allocation and trading rules. One can improve while the other worsens, so report both.

For a late-2015 signal, we may have a forecast and an open position but no completed 90-day return before the database ends. Exclude that row from prediction-error calculations, while continuing to include the holding in NAV. Report the number of evaluated dates and the last observed label so readers can see which part of the year contributed to the predictive results.

Monthly averages of 90-day targets also overlap. Compounding those averages as if they were consecutive one-month portfolio returns would count some of the same price movements more than once.

In [ ]:
prediction_report = []
for name, scores in [('fixed Ridge', fixed_scores), ('rolling Ridge', rolling_scores)]:
    evaluation = test_features[['date', 'symbol', 'target', 'label_end']].merge(scores, on=['date', 'symbol'], validate='one_to_one')
    evaluation = evaluation.loc[evaluation.date.ge('2015-01-01') & evaluation.target.notna()]
    prediction_report.append({'model': name, 'last_labeled_signal': evaluation.date.max(),
                              'last_label_end': evaluation.label_end.max(),
                              **prediction_metrics(evaluation, evaluation.score).to_dict()})
display(pd.DataFrame(prediction_report))

## 5. Explain the difference between forecast and portfolio results

The diagnostics below help locate where a strategy gained or lost money. A high average cash fraction means much of the capital earned no market return. `gross_traded_over_average_nav` measures the amount traded relative to the portfolio's typical size. Total fees show the direct cost of that trading, and remaining open lots show how much of the final NAV has not yet been realized in cash.

Compare these alongside the return metrics. If retraining raises IC but lowers NAV, check whether it changed only stocks outside the purchased group, bought repeated names while cash was scarce, or produced more expensive turnover. The fills and daily history let us investigate those explanations rather than infer them from a single score.

Next, rerun the fixed model at 0, 5, and 25 basis points per side without changing its forecasts or allocation rules. Higher fees reduce cash and can change the units available for later purchases, so the effect on terminal NAV need not equal a fixed fee subtraction from the original run. These scenarios measure sensitivity to assumed costs; they do not estimate actual market impact or trading capacity.

In [ ]:
diagnostics = []
for name, run in runs.items():
    h = run['history']
    diagnostics.append({'strategy': name, 'average_cash_fraction': (h.cash/h.nav).mean(),
                        'fees': h.fees.sum(), 'gross_traded_over_average_nav': h.gross_traded.sum()/h.nav.mean(),
                        'open_lots_at_end': len(run['open_lots']), 'stale_marks': int(h.stale_marks.sum())})
display(pd.DataFrame(diagnostics))
cost_rows = []
for bps in [0, 5, 25]:
    run = run_backtest(prices, benchmark, fixed_scores[['date', 'symbol', 'score']],
                       '2015-01-01', '2015-12-31', fee_bps=bps)
    cost_rows.append({'fee_bps': bps, 'fees': run['history'].fees.sum(), **performance(run['history'].nav).to_dict()})
display(pd.DataFrame(cost_rows))

## 6. Save the results

The next cell saves the experiment definition, including model hyperparameters, cross-validation scores, fit dates, predictions, NAV, fills, and remaining open lots under `Lectures/modeling/outputs/class_research/`. Together, these files let us trace a reported return back to the model and trades that produced it.

This output directory is ignored by Git. For HW4, commit the required small summaries and figures, and provide commands to regenerate the larger prediction and ledger files.

In [ ]:
output_dir = modeling_dir / 'outputs/class_research'
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / 'experiment.json').write_text(json.dumps(experiment, indent=2))
pd.DataFrame(search.cv_results_).to_csv(output_dir / 'cv_results.csv', index=False)
fit_audit.to_csv(output_dir / 'fit_audit.csv', index=False)
comparison.to_csv(output_dir / 'performance.csv')
for name, run in runs.items():
    stem = name.replace(' ', '_')
    run['history'].to_csv(output_dir / f'{stem}_nav.csv')
    run['fills'].to_csv(output_dir / f'{stem}_fills.csv', index=False)
    run['open_lots'].to_csv(output_dir / f'{stem}_open_lots.csv', index=False)
fixed_scores.to_csv(output_dir / 'fixed_predictions.csv', index=False)
rolling_scores.to_csv(output_dir / 'rolling_predictions.csv', index=False)
print('Saved experiment, validation, prediction, fit-audit, and accounting tables to:', output_dir)

## 7. Run on the full homework dataset

HW4 uses `equities_top2000.duckdb`, containing 3,849 names selected across quarterly universes, including 1,355 marked inactive in the source snapshot. Read the [assignment](hw4.md) and [data guide](../modeling/README.md) before changing the input path. Use `universe_membership` for each entry quarter; stocks selected later are not automatically candidates on earlier dates.

Develop on the class extract first. For the full dataset, select only the needed columns and dates in SQL, and process tickers in batches with enough warm-up history for their features. Build daily features before sampling weekly or month-end training dates. Each entry signal still needs scores for the full eligible cross-section, and every open holding needs a mark on each market session.

Measure data loading, feature calculation, fitting, and simulation separately to find what takes time. Start with a small model search and limited parallelism. The full dataset ends on September 30, 2026; open positions remain in final NAV. Incomplete delisting records may leave a held stock without quotes, in which case the missing-price policy can stop the run and require investigation.

## Exercises

### Your turn: reconstruct the cash balance

Using the fixed Ridge fills, compute `your_reconstructed_cash` from $1,000,000 plus cumulative daily `cash_change`, aligned to every history date. Compare it with the engine's cash column only after calculating your result.

In [ ]:
your_reconstructed_cash = None

In [ ]:
if your_reconstructed_cash is None:
    print('Group fill cash changes by date, reindex missing days to zero, then cumulatively sum.')
else:
    h = runs['fixed Ridge']['history']
    pd.testing.assert_index_equal(your_reconstructed_cash.index, h.index)
    np.testing.assert_allclose(your_reconstructed_cash, h.cash, atol=1e-7)
    print('Independent cash reconstruction passed.')

### Your turn: audit one rolling fit

For the 2015-06-30 fit, count eligible candidate rows since 2010 with signal dates before the fit but label ends on/after it or unknown. Store the count in `your_immature_rows` and explain in `your_retraining_note` why these rows cannot be used even though their features already existed.

In [ ]:
your_immature_rows = None
your_retraining_note = ''

In [ ]:
if your_immature_rows is None:
    print('An observed feature row can still have an unobserved target.')
else:
    cutoff = pd.Timestamp('2015-06-30')
    candidates = panel.loc[panel.eligible & panel.date.ge('2010-01-01') & panel.date.lt(cutoff)]
    expected = (candidates.label_end.ge(cutoff) | candidates.label_end.isna()).sum()
    assert your_immature_rows == expected and your_retraining_note.strip()
    print('Retraining audit passed. Every fit needs this same maturity check.')

### Your turn: write the research conclusion

In 150–250 words, explain what the prediction metrics and portfolio results tell you about the model. Identify one limitation that code cannot repair and recommend the next experiment to run on development data. Discuss an adverse result if there is one.